In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from scipy.optimize import fmin_l_bfgs_b
from typing import Tuple
from utils.utils import load_iris, split_db_2to1, load_iris_binary
from utils.BinaryLR import BinaryLR

# Gradient Descend

In [2]:
def f(x: np.ndarray) -> float:
    return np.power(x[0]+3, 2) + np.sin(x[0]) + np.power(x[1]-1, 2)

def f_with_grad(x: np.ndarray) -> Tuple[float, np.ndarray]:
    y = np.power(x[0]+3, 2) + np.sin(x[0]) + np.power(x[1]-1, 2)
    grad = np.zeros_like(x)
    grad[0] = 2*(x[0]+3) + np.cos(x[0])
    grad[1] = 2*(x[1]-1)
    return y, grad

def grad(X: np.ndarray) -> np.ndarray:
    eps = 1e-7
    gradients = np.zeros_like(X)
    for i, Xi in enumerate(X):
        ei = np.zeros_like(X)
        ei[i] = 1
        gradients[i] = (f(X + eps*ei) - f(X - eps*ei)) / (2*eps)
    return gradients

In [3]:
# Approx Grad

x0 = np.array([0.0, 0.0])
x_opt, f_opt, d = fmin_l_bfgs_b(func=f, x0=x0, approx_grad=True)
print("Optimal point:", x_opt)
print("Function value at optimal point:", f_opt)
print("Optimality measure:", d)


Optimal point: [-2.57747138  0.99999927]
Function value at optimal point: -0.3561430123647448
Optimality measure: {'grad': array([-1.51545444e-06, -1.45439215e-06]), 'task': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', 'funcalls': 21, 'nit': 6, 'warnflag': 0}


In [4]:
# Own generic grad

x0 = np.array([0.0, 0.0])
x_opt, f_opt, d = fmin_l_bfgs_b(func=f, x0=x0, fprime=grad, approx_grad=False)
print("Optimal point:", x_opt)
print("Function value at optimal point:", f_opt)
print("Optimality measure:", d)

Optimal point: [-2.57747137  0.99999927]
Function value at optimal point: -0.35614301236476137
Optimality measure: {'grad': array([-1.50296442e-06, -1.46105350e-06]), 'task': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', 'funcalls': 7, 'nit': 6, 'warnflag': 0}


In [5]:
# Own grad

x0 = np.array([0.0, 0.0])
x_opt, f_opt, d = fmin_l_bfgs_b(func=f_with_grad, x0=x0, approx_grad=False)
print("Optimal point:", x_opt)
print("Function value at optimal point:", f_opt)
print("Optimality measure:", d)

Optimal point: [-2.57747137  0.99999927]
Function value at optimal point: -0.3561430123647611
Optimality measure: {'grad': array([-1.50318729e-06, -1.46120529e-06]), 'task': 'CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', 'funcalls': 7, 'nit': 6, 'warnflag': 0}


# Binary Logistic Regression

In [6]:
D, L = load_iris_binary()
(DTR, LTR), (DVAL, LVAL) = split_db_2to1(D, L)

In [7]:
model = BinaryLR(lamb=1, pi1=0.5)
model.fit(DTR, LTR)
pred = model.predict(DVAL)

Optimal point: [-0.11040209 -0.0289869  -0.2478711  -0.14950473  2.3109451 ]
Function value at optimal point: 0.6316436205354086


In [8]:
np.sum(pred == LVAL) / len(LVAL)

np.float64(0.8529411764705882)

In [9]:
print(model.evaluate(LVAL)[0])

0.1470588235294118


In [10]:
(err, cm, DCFu, DCF) = model.evaluate(LVAL)
minDCF = model.calc_minDCF(LVAL)

print("Error Rate:", err)
print("Actual DCF:", DCF)
print("Min DCF:", minDCF)

Error Rate: 0.1470588235294118
Actual DCF: 0.2777777777777778
Min DCF: 0.1111111111111111
